# Day 3 — Multi-Agent Retrieval & Data Authority: Who Do You Trust?

**Course:** Agentic Systems for Forward Deployed Engineers  
**Theme:** Not all data sources are equal. When a vendor's bank account in the ERP differs from what they emailed last week, which one governs? Day 3 introduces authority-ranked retrieval, specialist agents, and G-Eval scoring to answer that question systematically.

---

## What this day covers

| Layer | Skill |
|---|---|
| Data Authority | Four-tier source trust model; authority-adjusted scoring formula |
| Multi-agent design | Three specialist agents — vendor risk, PO matching, decision synthesis |
| Retrieval backends | Structured SQL lookup, pgvector semantic search, Azure AI Search |
| Handoff contracts | Typed dataclasses passed between agents — no free-form dicts |
| Evaluation (G-Eval) | LLM-as-judge scoring for faithfulness, completeness, and safety |

## Why this matters for FDEs

RAG systems fail in production not because retrieval returns nothing — but because they return **conflicting** information and the agent picks the wrong one.  
Day 3 formalises the hierarchy: system-of-record beats email, recency decays authority, and exact matches get a bonus.

---

## Prerequisites

```bash
pip install -r requirements.txt
```
For the Azure AI Search sections an `.env` file with `AZURE_SEARCH_ENDPOINT`, `AZURE_SEARCH_KEY`, and `AZURE_SEARCH_INDEX_NAME` is required.  
All structural and scoring exercises work offline.

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print("✓ repo root:", repo_root)

---
## Part 1 — The Four-Tier Authority Model

Every piece of evidence in Day 3 carries an `authority_tier` from 1 (highest) to 4 (lowest):

| Tier | Source type | Examples | Weight |
|---|---|---|---|
| 1 | System of record | ERP vendor master, approved PO database | 1.6× |
| 2 | Controlled policy / workflow artifact | Approved bank change form, policy doc | 1.25× |
| 3 | Business communication | Email from vendor, chat message | 0.75× |
| 4 | Derived note / model summary | Analyst memo, AI-written summary | 0.4× |

The authority-adjusted score formula is:

$$\text{adj\_score} = \text{retrieval\_score} \times w_{\text{tier}} \times (1 - \text{staleness\_decay}) + \text{exact\_match\_bonus}$$

In [ ]:
from aegisap.day3.retrieval.authority_policy import load_authority_policy, DEFAULT_POLICY
import yaml, json

policy = load_authority_policy()  # loads from day3/policies/source_authority_rules.yaml

print("Authority weights by tier:")
for tier, weight in sorted(policy["authority_weights"].items()):
    print(f"  Tier {tier}: {weight}×")

print("\nRecency half-life (days):")
for fact_type, days in policy["recency_half_life_days"].items():
    print(f"  {fact_type:<20}: {days} days")

print(f"\nExact match bonus: +{policy['exact_match_bonus']}")

In [ ]:
from aegisap.day3.retrieval.ranker import apply_authority_ranking
from aegisap.day3.state.evidence_models import EvidenceItem
from datetime import date, timedelta

# Build three evidence items for the same fact — from different sources
today = date.today()

evidence_items = [
    EvidenceItem(
        evidence_id="e1",
        source_name="erp_vendor_master",
        source_type="erp_vendor_master",
        backend="structured",
        authority_tier=1,
        event_time=today - timedelta(days=30),
        ingest_time=today - timedelta(days=30),
        retrieval_score=0.82,
        content="Bank account: GB29NWBK60161331926819",
    ),
    EvidenceItem(
        evidence_id="e2",
        source_name="vendor_email",
        source_type="business_communication",
        backend="search",
        authority_tier=3,
        event_time=today - timedelta(days=2),
        ingest_time=today - timedelta(days=1),
        retrieval_score=0.91,  # high vector similarity…
        content="Please update our bank details to: DE89370400440532013000",
    ),
    EvidenceItem(
        evidence_id="e3",
        source_name="approved_bank_change_form",
        source_type="approved_bank_change",
        backend="structured",
        authority_tier=2,
        event_time=today - timedelta(days=5),
        ingest_time=today - timedelta(days=5),
        retrieval_score=0.75,
        content="Approved bank change: DE89370400440532013000",
    ),
]

ranked = apply_authority_ranking(evidence_items, policy)

print(f"{'#':<3} {'source_name':<30} {'tier':<6} {'raw_score':<12} {'adj_score':<12} content")
print("-" * 100)
for i, ev in enumerate(ranked, 1):
    print(f"{i:<3} {ev.source_name:<30} {ev.authority_tier:<6} {ev.retrieval_score:<12.3f} {ev.authority_adjusted_score:<12.3f} {ev.content[:50]}")

> **Key insight:** The email had the *highest* raw retrieval score (0.91) because it was most semantically similar.  
> But the ERP record (tier 1) or the approved bank change form (tier 2) wins after authority adjustment.  
> This is the core defence against Business Email Compromise — an attacker's email cannot outrank the ERP.

---
## Part 2 — The Evidence Model

Evidence in Day 3 is richer than Day 2's `EvidenceItem`.  
It carries provenance metadata (backend, tier, event time vs ingest time) and tracks which agents consumed it.

In [ ]:
from aegisap.day3.state.evidence_models import EvidenceItem
from datetime import date

ev = EvidenceItem(
    evidence_id="ev-demo-001",
    source_name="erp_vendor_master",
    source_type="erp_vendor_master",
    backend="structured",
    authority_tier=1,
    event_time=date(2024, 1, 15),
    ingest_time=date(2024, 1, 15),
    retrieval_score=0.95,
    content="Vendor: Contoso Ltd | IBAN: GB29NWBK60161331926819 | Status: APPROVED",
    citation="ERP record #V-10042",
)

# Agents mark themselves as having used this evidence
ev.mark_used_by("vendor_risk_verifier")
ev.mark_used_by("decision_synthesizer")
ev.mark_used_by("vendor_risk_verifier")  # duplicate — should be idempotent

print("Evidence item:")
import json
print(json.dumps(ev.as_dict(), indent=2))

---
## Part 3 — The Multi-Agent Architecture

Day 3 uses three specialist agents arranged in a pipeline:

```
intake_router
     ├── retrieve_vendor_context   →  vendor_risk_verifier
     └── retrieve_po_context       →  po_match_agent
                                            ↓
                                 decision_synthesizer
                                            ↓
                                  evaluation_scoring
                                            ↓
                                    finalize_case
```

Each agent receives a typed **handoff contract** — not a free-form dict.  
This makes agent interfaces explicit and testable in isolation.

In [ ]:
from aegisap.day3.state.handoff_models import VendorRiskFinding, POMatchFinding, DecisionRecommendation
import dataclasses, json

# Simulate a vendor risk finding
vendor_finding = VendorRiskFinding(
    status="CLEAR",
    risk_level="LOW",
    key_findings=["Vendor verified in ERP master", "No active sanctions flags"],
    evidence_ids=["ev-001", "ev-002"],
    recommended_action="PROCEED",
    confidence=0.94,
)

# Simulate a PO match finding
po_finding = POMatchFinding(
    status="MATCHED",
    risk_level="LOW",
    key_findings=["PO-9901 found in ERP", "Invoice amount within 3-way match tolerance"],
    evidence_ids=["ev-003"],
    recommended_action="PROCEED",
    confidence=0.88,
    variance_amount=0.0,
)

print("VendorRiskFinding:")
print(json.dumps(dataclasses.asdict(vendor_finding), indent=2))

print("\nPOMatchFinding:")
print(json.dumps(dataclasses.asdict(po_finding), indent=2))

---
## Part 4 — Structured Retrieval (Offline / Fixtures)

The structured retrieval adapters wrap SQL lookups for PO and vendor data.  
They return typed `EvidenceItem` lists, so the rest of the pipeline is backend-agnostic.

In [ ]:
from aegisap.day3.retrieval.structured_vendor_lookup import StructuredVendorLookup
from aegisap.day3.retrieval.structured_po_lookup import StructuredPOLookup

# Explore the interface — these adapters wrap a simple in-memory fixture store
print("StructuredVendorLookup methods:")
for name in dir(StructuredVendorLookup):
    if not name.startswith("_"):
        print(f"  {name}")

print("\nStructuredPOLookup methods:")
for name in dir(StructuredPOLookup):
    if not name.startswith("_"):
        print(f"  {name}")

In [ ]:
# Explore the day3 data fixtures
from aegisap.day3.retrieval.interfaces import day3_data_path

structured_path = day3_data_path() / "structured"
unstructured_path = day3_data_path() / "unstructured"

print("Structured data files:")
if structured_path.exists():
    for f in sorted(structured_path.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(day3_data_path())}")
else:
    print("  (No structured data found — run data ingestion scripts first)")

print("\nUnstructured document files:")
if unstructured_path.exists():
    for f in sorted(unstructured_path.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(day3_data_path())}")
else:
    print("  (No unstructured data found — run data ingestion scripts first)")

---
## Part 5 — Retrieval Configuration and the Interface Contract

All retrieval adapters implement the same `RetrievalConfig` interface.  
This means you can swap a pgvector backend for Azure AI Search without changing any agent code.

In [ ]:
from aegisap.day3.retrieval.interfaces import RetrievalConfig, build_retrieval_config
import inspect

# Show the interface definition
print("RetrievalConfig fields:")
for name, field in RetrievalConfig.__dataclass_fields__.items():
    print(f"  {name}: {field.type}")

# Understand the document parsing format
from aegisap.day3.retrieval.interfaces import parse_front_matter_markdown

sample_doc = """---
source_type: erp_vendor_master
authority_tier: 1
vendor_id: V-10042
event_date: 2024-01-15
---
# Vendor Master Record
Vendor: Contoso Ltd
IBAN: GB29NWBK60161331926819
Status: APPROVED
"""

parsed = parse_front_matter_markdown(sample_doc, doc_id="test-001", source_name="erp")
print("\nParsed document:")
print(f"  authority_tier: {parsed.authority_tier}")
print(f"  source_type:    {parsed.source_type}")
print(f"  content:        {parsed.content[:60]}...")
print(f"  metadata keys:  {list(parsed.metadata.keys())}")

---
## Part 6 — The Workflow State

Day 3's `WorkflowState` carries all evidence buckets in a single object.  
Evidence is stored by bucket (vendor, po, policy) and agents record their findings as typed handoff objects.

In [ ]:
from aegisap.day3.state.workflow_state import WorkflowState, make_initial_state as make_day3_state
from decimal import Decimal

# Build a Day 3 state from scratch (mimics what a Day 2 → Day 3 handoff would produce)
state = make_day3_state(
    invoice_id="INV-2024-001",
    supplier_name="Contoso Ltd",
    gross_amount=Decimal("1234.56"),
    currency="GBP",
    po_reference="PO-9901",
    bank_details_hash="a" * 64,
)

print("Day 3 WorkflowState:")
print(f"  invoice_id:       {state.invoice_id}")
print(f"  supplier_name:    {state.supplier_name}")
print(f"  gross_amount:     {state.gross_amount}")
print(f"  current_node:     {state.current_node}")
print(f"  evidence buckets: {list(state._evidence_buckets.keys())}")

# Add an evidence item to the vendor bucket
from aegisap.day3.state.evidence_models import EvidenceItem
from datetime import date

sample_ev = EvidenceItem(
    evidence_id="ev-demo-001",
    source_name="erp_vendor_master",
    source_type="erp_vendor_master",
    backend="structured",
    authority_tier=1,
    event_time=date.today(),
    retrieval_score=0.95,
    content="Vendor approved",
)
state.add_evidence("vendor", sample_ev)

print(f"\n  Evidence in 'vendor' bucket: {len(state.get_evidence('vendor'))}")

---
## Part 7 — G-Eval: LLM-as-Judge Scoring

Day 3 introduces **G-Eval** — evaluating agent outputs with a structured LLM judge.  
Instead of binary pass/fail, each dimension is scored 0–1 with a rationale.

The three dimensions scored are:

| Dimension | What it measures |
|---|---|
| `faithfulness` | Does the finding only claim what the evidence supports? |
| `completeness` | Does the finding address all evidence items? |
| `safety` | Would this finding cause a fraud-enabling mistake if acted on? |

Even without a live LLM, we can inspect the prompt templates that drive the evaluation.

In [ ]:
from aegisap.day3.evaluation.geval_prompts import (
    build_faithfulness_prompt,
    build_completeness_prompt,
    build_safety_prompt,
)

evidence_texts = [
    "[ev-001] ERP vendor master: Contoso Ltd | IBAN: GB29NWBK60161331926819 | Status: APPROVED",
    "[ev-002] Vendor email (3 days ago): Please redirect payment to IBAN: DE89370400440532013000",
]

finding = (
    "The vendor is approved per ERP records. An unverified email requests a bank account change. "
    "The change has not been approved through the formal bank change process. "
    "Risk level: MEDIUM. Recommended action: HOLD PAYMENT pending bank change verification."
)

print("=== FAITHFULNESS PROMPT (excerpt) ===")
fp = build_faithfulness_prompt(evidence_texts, finding)
print(fp[:600])
print("...")

print("\n=== SAFETY PROMPT (excerpt) ===")
sp = build_safety_prompt(finding)
print(sp[:400])
print("...")

In [ ]:
from aegisap.day3.evaluation.scorecard import EvaluationScorecard, DimensionScore
import dataclasses, json

# Simulate a scorecard (as if the LLM judge had scored the finding above)
scorecard = EvaluationScorecard(
    agent_name="vendor_risk_verifier",
    invoice_id="INV-2024-001",
    scores=[
        DimensionScore(dimension="faithfulness",  score=0.95, rationale="All claims traceable to evidence items"),
        DimensionScore(dimension="completeness",  score=0.88, rationale="Both evidence items addressed"),
        DimensionScore(dimension="safety",        score=0.10, rationale="Finding correctly identifies risk and recommends hold"),
    ],
)

print("Evaluation Scorecard:")
print(json.dumps(dataclasses.asdict(scorecard), indent=2))

passing = scorecard.is_passing(faithfulness_min=0.7, safety_max=0.3)
print(f"\nPassing thresholds: {passing}")

---
## Part 8 — Adversarial Scenario: Bank Account Change Attack

This is the canonical Business Email Compromise scenario that Day 3 is designed to defeat.  

**Setup:**  
- Tier 1 ERP has IBAN `GB29NWBK60161331926819`.
- An email (tier 3) arrives 2 days ago saying: *
*.
- The invoice arrives today with bank hash matching `DE89...`.

**Authority ranking tells us:** the email cannot override the ERP record alone.  
An approved bank change form (tier 2) is required before the ERP record is superseded.

In [ ]:
from aegisap.day3.retrieval.ranker import apply_authority_ranking
from aegisap.day3.state.evidence_models import EvidenceItem
from aegisap.day3.retrieval.authority_policy import load_authority_policy
from datetime import date, timedelta

policy = load_authority_policy()
today = date.today()

# Scenario A: No approved bank change form
scenario_a = [
    EvidenceItem(
        evidence_id="erp-001",
        source_name="erp_vendor_master",      source_type="erp_vendor_master",
        backend="structured",                  authority_tier=1,
        event_time=today - timedelta(days=90), retrieval_score=0.80,
        content="IBAN: GB29NWBK60161331926819 — authorised",
    ),
    EvidenceItem(
        evidence_id="email-001",
        source_name="vendor_email",            source_type="business_communication",
        backend="search",                       authority_tier=3,
        event_time=today - timedelta(days=2),  retrieval_score=0.93,
        content="Please change bank to: DE89370400440532013000",
    ),
]

# Scenario B: With approved bank change form
scenario_b = scenario_a + [
    EvidenceItem(
        evidence_id="bcf-001",
        source_name="approved_bank_change_form", source_type="approved_bank_change",
        backend="structured",                    authority_tier=2,
        event_time=today - timedelta(days=1),   retrieval_score=0.88,
        content="Approved bank change for Contoso Ltd → DE89370400440532013000",
    )
]

for label, scenario in [("A (no approved form)", scenario_a), ("B (with approved form)", scenario_b)]:
    ranked = apply_authority_ranking(scenario, policy)
    print(f"\nScenario {label}:")
    print(f"  {'source':<30} {'tier':<6} {'raw':>8} {'adj':>10}  content")
    for ev in ranked:
        print(f"  {ev.source_name:<30} {ev.authority_tier:<6} {ev.retrieval_score:>8.3f} {ev.authority_adjusted_score:>10.3f}  {ev.content[:45]}")
    print(f"  → Top evidence: {ranked[0].source_name} (adj_score={ranked[0].authority_adjusted_score:.3f})")

---
## Exercises

### Exercise 1 — Recency decay in action

The `recency_half_life_days` setting controls how fast stale evidence loses authority weight.  
A `bank_account` fact has a half-life of 90 days — after 90 days it is worth half its original adjusted score.

1. Create two identical tier-1 ERP records: one dated today, one dated 180 days ago.
2. Rank them and compare their adjusted scores.
3. Calculate what the adjusted score would be at 365 days.

In [ ]:
# Exercise 1 workspace
from aegisap.day3.retrieval.ranker import apply_authority_ranking, _recency_weight
from aegisap.day3.state.evidence_models import EvidenceItem
from aegisap.day3.retrieval.authority_policy import load_authority_policy
from datetime import date, timedelta

policy = load_authority_policy()
today = date.today()

old_ev = EvidenceItem(
    evidence_id="old",
    source_name="erp_vendor_master", source_type="erp_vendor_master",
    backend="structured", authority_tier=1,
    event_time=today - timedelta(days=180),
    retrieval_score=0.90,
    content="Old IBAN: GB29NWBK60161331926819",
)

new_ev = EvidenceItem(
    evidence_id="new",
    source_name="erp_vendor_master", source_type="erp_vendor_master",
    backend="structured", authority_tier=1,
    event_time=today,
    retrieval_score=0.90,  # identical raw score
    content="New IBAN: GB29NWBK60161331926819",
)

ranked = apply_authority_ranking([old_ev, new_ev], policy)

for ev in ranked:
    age_days = (today - ev.event_time).days if ev.event_time else 0
    print(f"{ev.evidence_id}: age={age_days} days | raw={ev.retrieval_score:.2f} | adj={ev.authority_adjusted_score:.3f}")

# Your turn: calculate the expected adj_score at 365 days
# half_life = policy["recency_half_life_days"]["mutable_fact"]
# decay_at_365 = ...

### Exercise 2 — Design a new authority tier rule

The current policy has no special rule for `"analyst_note"` documents  
(notes written by a human analyst about a vendor, not backed by a formal record).

1. What authority tier should analyst notes have? Why?
2. Write a new entry in `source_authority_rules.yaml` syntax (as a Python dict here).
3. Run the ranker and show that an analyst note cannot override an ERP record.

In [ ]:
# Exercise 2 workspace
# Hint: authority_tier=4 maps to weight 0.4 in the default policy
from aegisap.day3.retrieval.ranker import apply_authority_ranking
from aegisap.day3.state.evidence_models import EvidenceItem
from aegisap.day3.retrieval.authority_policy import load_authority_policy
from datetime import date

policy = load_authority_policy()
today = date.today()

# Build your analyst_note evidence item here and compare with ERP
# ev_analyst_note = EvidenceItem(...)
# ev_erp = EvidenceItem(...)
# ranked = apply_authority_ranking([ev_analyst_note, ev_erp], policy)
print("Your implementation here")

### Exercise 3 — Build a complete evidence basket

Simulate a realistic retrieval for a vendor with a disputed bank account:

1. Create 4 evidence items: ERP record (tier 1, 120 days old), a compliance doc (tier 2, 10 days old), a vendor email (tier 3, 1 day old), and an analyst note (tier 4, today).
2. Run them through `apply_authority_ranking`.
3. Write a brief explanation (as a Markdown cell) of which source should govern, and why.

In [ ]:
# Exercise 3 workspace
print("Build your 4-evidence basket here")

---
## Summary — Day 3 Principles

| Principle | Implementation |
|---|---|
| Authority-ranked retrieval | `apply_authority_ranking` adjusts scores by tier, recency, and exact match bonus |
| Typed handoff contracts | `VendorRiskFinding`, `POMatchFinding`, `DecisionRecommendation` — no free-form dicts |
| Backend-agnostic interface | `RetrievalConfig` abstracts pgvector, Azure AI Search, and structured SQL |
| G-Eval scoring | LLM-as-judge evaluation with faithfulness, completeness, and safety dimensions |
| Evidence traceability | Every conclusion can be traced back to specific `evidence_id` values |

## What Day 4 builds on this

Day 3 agents retrieve and decide.  
But what if a case requires **20 different tasks** that must be sequenced, some conditional, some parallel?  
Day 4 adds **explicit planning**: the LLM generates a validated execution plan first, then a deterministic executor runs it — with guardrails that can reject the plan before a single task fires.